In [1]:
import pandas as pd
import torch
import torch.nn as nn

In [2]:
df = pd.read_csv("movie review/train.csv")

In [3]:
df.shape

(25000, 2)

In [4]:
sentences = df['text']

labels = df['sentiment'].map({'neg':0,'pos':1})

In [5]:
# tokenizer & vocabulary
import re
def tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]","",text)
    return text.split()
    

# build vocab
vocab = {'<PAD>': 0, '<UNK>': 1}
idx = 2

for sentence in sentences:
    for word in tokenizer(sentence):
        if word not in vocab:
            vocab[word] = idx
            idx +=1

In [6]:
# padding 
max_len = 200

# convert text to vector
def encode(sentence):
    tokens = tokenizer(sentence)
    tokens = tokens[:max_len]
    return [vocab.get(word,vocab['<UNK>']) for word in tokens]

encoded_sentences = [encode(s) for s in sentences]

padded = []

for s in encoded_sentences:
    s = s + [0] * (max_len- len(s))
    padded.append(s)

X = torch.tensor(padded)
y = torch.tensor(labels).float().unsqueeze(1)

In [7]:
# lstm model
class SentimentLSTM(nn.Module):
    def __init__(self,vocab_size,embed_size,hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,embed_size)
        self.lstm = nn.LSTM(embed_size,hidden_size,batch_first=True,bidirectional=True)
        self.fc = nn.Linear(hidden_size*2,1)
        self.sigmoid=nn.Sigmoid()
        self.dropout = nn.Dropout(0.3)

    def forward(self,x):
        x = self.embedding(x)
        x = self.dropout(x)
        out,_ = self.lstm(x)
        # out = out[:, -1, :] 
        out = torch.mean(out, dim=1)
        out = self.fc(out)
        return out
        

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [9]:
model = SentimentLSTM(vocab_size=len(vocab)+1,embed_size=128,hidden_size=256).to(device)

# loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

X= X.to(device)
y = y.to(device)

In [10]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, y)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

batch_size = 64 

In [11]:
import time
best_loss = float('inf')
patience = 3
counter = 0
epochs = 30
for epoch in range(0,epochs):
    model.train()
    epoch_loss = 0
    start_time = time.time()
    
    for i,(batch_X,batch_y) in enumerate(loader):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        output = model(batch_X)
        # loss
        loss = criterion(output,batch_y)
        # optimizer
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss +=loss.item()

        # batch wise print
        if i  % 100 ==0 and i >0:
            print(f"Epoch [{epoch+1}/{epochs}],Batch [{i+1}], Loss: {loss.item():.4f}")
    # Average epoch loss
    avg_loss = epoch_loss / len(loader)
    epoch_duration = time.time() - start_time
    print(f"==> Epoch [{epoch+1}/{epochs}] Average Loss: {avg_loss:.4f}, Time: {epoch_duration:.1f}s")
    print("-"*50)

    # early stopping
    if avg_loss< best_loss:
        best_loss= avg_loss
        counter = 0
        print('Loss Improved')

        # save the best model
        torch.save(model.state_dict(),'best_model.pt')
    else:
        counter +=1
        print(f"No Improvement({counter}/{patience})")

        if counter >= patience:
            print("Early Stopping Triggred")
            total_duration = start_time- time.time()
            print(total_duration)
            break

Epoch [1/30],Batch [101], Loss: 0.5376
Epoch [1/30],Batch [201], Loss: 0.5205
Epoch [1/30],Batch [301], Loss: 0.4742
Epoch [1/30],Batch [401], Loss: 0.5589
Epoch [1/30],Batch [501], Loss: 0.4718
Epoch [1/30],Batch [601], Loss: 0.4661
Epoch [1/30],Batch [701], Loss: 0.4327
==> Epoch [1/30] Average Loss: 0.5066, Time: 33.4s
--------------------------------------------------
Loss Improved
Epoch [2/30],Batch [101], Loss: 0.3687
Epoch [2/30],Batch [201], Loss: 0.3433
Epoch [2/30],Batch [301], Loss: 0.4367
Epoch [2/30],Batch [401], Loss: 0.5597
Epoch [2/30],Batch [501], Loss: 0.2848
Epoch [2/30],Batch [601], Loss: 0.2243
Epoch [2/30],Batch [701], Loss: 0.3039
==> Epoch [2/30] Average Loss: 0.3140, Time: 30.4s
--------------------------------------------------
Loss Improved
Epoch [3/30],Batch [101], Loss: 0.1019
Epoch [3/30],Batch [201], Loss: 0.3373
Epoch [3/30],Batch [301], Loss: 0.1654
Epoch [3/30],Batch [401], Loss: 0.1671
Epoch [3/30],Batch [501], Loss: 0.1827
Epoch [3/30],Batch [601], L

In [22]:
def predict(sentence):
    encoded = encode(sentence)
    encoded = encoded + [0]*(max_len - len(encoded))
    
    tensor = torch.tensor([encoded]).to(device)   #move to GPU
    
    
    model.eval()
    with torch.no_grad():
        pred = model(tensor)
        prob = torch.sigmoid(pred)
    print("Raw:", prob.item())
    return "Positive" if pred.item() > 0 else "Negative"

In [30]:
def predict(sentence):
    encoded = encode(sentence)
    encoded = encoded + [0]*(max_len - len(encoded))
    tensor = torch.tensor([encoded]).to(device)
    
    model.eval()
    with torch.no_grad():
        logit = model(tensor)
        prob = torch.sigmoid(logit)
    
    # This helps you debug better:
    print(f"Logit: {logit.item():.4f} | Prob: {prob.item():.4f}")
    
    return "Positive" if logit.item() > 0 else "Negative"

# Testing on Dataset

In [31]:
# load the test datset
test_df = pd.read_csv('movie review/test.csv')
test_df.head()

,text,sentiment
0,"My daughter liked it but I was aghast, that a ...",neg
1,I... No words. No words can describe this. I w...,neg
2,this film is basically a poor take on the old ...,neg
3,"This is a terrible movie, and I'm not even sur...",neg
4,First of all this movie is a piece of reality ...,pos


In [32]:
test_sentence = test_df['text']
test_labels = test_df['sentiment'].map({'neg':0,'pos':1})

# encoding and padding
encoded_test = [encode(s) for s in test_sentence]
padded_test = []

for s in encoded_test:
    s = s + [0] * (max_len - len(s))
    padded_test.append(s)
    

In [33]:
#convert to tensors
test_X = torch.tensor(padded_test).to(device)
test_y = torch.tensor(test_labels).float().unsqueeze(1).to(device)

In [34]:
# create dataloder for testing
test_dataset = TensorDataset(test_X,test_y)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [35]:
# Evaluation
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_X,batch_y in test_loader:
        output = model(batch_X)

        predictions = (output> 0).float()

        total += batch_y.size(0)
        correct += (predictions==batch_y).sum().item()

accuracy = (correct/ total) * 100
print(f"Test Accuracy: {accuracy:.2f} ")

Test Accuracy: 85.22 


In [36]:
# Load the winner!
# Add weights_only=True
model.load_state_dict(torch.load('best_model.pt', weights_only=True))
model.eval()

# Now predict
print(predict("This was an absolute masterpiece!"))

Logit: 5.9015 | Prob: 0.9973
Positive


In [39]:
#  Demo Script
test_reviews = [
    "A cinematic masterpiece that everyone should see.",
    "I fell asleep halfway through. Boring and long.",
    "The acting was great but the script was terrible.",
    "Don't listen to the critics, this movie is a gem!",
    "I've never seen anything quite so bad in my life.",
    "The visuals were stunning, but it lacked any soul.",
    "Absolutely hilarious from start to finish!",
    "Waste of talent, waste of time, waste of money.",
    "Not as bad as people say, but still not good.",
    "I would rather watch paint dry than see this again."
]

for r in test_reviews:
    print(f"Review: {r}")
    print(f"Result: {predict(r)}")
    print("-" * 30)

Review: A cinematic masterpiece that everyone should see.
Logit: 8.1309 | Prob: 0.9997
Result: Positive
------------------------------
Review: I fell asleep halfway through. Boring and long.
Logit: -8.7014 | Prob: 0.0002
Result: Negative
------------------------------
Review: The acting was great but the script was terrible.
Logit: -9.6115 | Prob: 0.0001
Result: Negative
------------------------------
Review: Don't listen to the critics, this movie is a gem!
Logit: 10.7389 | Prob: 1.0000
Result: Positive
------------------------------
Review: I've never seen anything quite so bad in my life.
Logit: -1.8581 | Prob: 0.1349
Result: Negative
------------------------------
Review: The visuals were stunning, but it lacked any soul.
Logit: 0.2740 | Prob: 0.5681
Result: Positive
------------------------------
Review: Absolutely hilarious from start to finish!
Logit: 6.6396 | Prob: 0.9987
Result: Positive
------------------------------
Review: Waste of talent, waste of time, waste of money.
Log